In [ ]:
import sys

sys.path.append("..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from AnimalCLEF_baselines.metrics import score_2025, score_2026
from AnimalCLEF_baselines.submissions import evaluate_submissions, get_submission, load_submissions, select_best
from matplotlib.ticker import FuncFormatter

In [ ]:
def count_individuals_true(solution: pd.DataFrame) -> dict:
    return solution.groupby("dataset")["identity"].nunique().to_dict()


def overshoot_individuals_2026(submission: pd.DataFrame, solution: pd.DataFrame) -> dict:
    n_true = count_individuals_true(solution)
    clusters = pd.Series(submission["cluster"].to_numpy(), index=solution.index)
    n_pred = clusters.groupby(solution["dataset"]).nunique().to_dict()
    return {dataset: n_pred[dataset] / n_true[dataset] for dataset in n_pred}


def overshoot_individuals_2025(submission: pd.DataFrame, solution: pd.DataFrame) -> dict:
    n_true = count_individuals_true(solution)
    predictions = submission["identity"].to_numpy()
    overshoot = {}
    for dataset in solution["dataset"].unique():
        mask = (solution["dataset"] == dataset).to_numpy()
        predictions_dataset = predictions[mask]
        is_known = predictions_dataset != "new_individual"
        n_known = is_known.sum()
        if n_known == 0:
            overshoot[dataset] = np.nan
            continue
        n_returning = len(set(predictions_dataset[is_known]))
        n_pred = n_returning * len(predictions_dataset) / n_known
        overshoot[dataset] = n_pred / n_true[dataset]
    return overshoot


def overshoot_newcomers_2025(submission: pd.DataFrame, solution: pd.DataFrame) -> dict:
    solution_new = solution[solution["new_identity"]]
    n_true = solution_new.groupby("dataset")["identity"].nunique().to_dict()

    predictions = submission["identity"].to_numpy()
    overshoot = {}
    for dataset in solution["dataset"].unique():
        mask = (solution["dataset"] == dataset).to_numpy()
        predictions_dataset = predictions[mask]
        is_known = predictions_dataset != "new_individual"
        n_known = is_known.sum()
        if n_known == 0:
            overshoot[dataset] = np.nan
            continue
        n_returning = len(set(predictions_dataset[is_known]))
        n_new = (~is_known).sum()
        n_pred = n_new * n_returning / n_known
        overshoot[dataset] = n_pred / n_true[dataset]
    return overshoot

In [ ]:
editions = {
    "AnimalCLEF2025": {
        "root": "/data/wildlife_datasets/data/AnimalCLEF2025",
        "score": score_2025,
        "metric": "normalized",
        "replace": {"0009": "XiangYuhang", "0011": "XuZhe"},
        "count_fn": overshoot_individuals_2025,
        "plot_fn": overshoot_newcomers_2025,
        "plot_name": "AnimalCLEF2025: open-set",
        "score_col": "{}_normalized",
        "y_label": "ratio of predicted/true newcomers",
    },
    "AnimalCLEF2026": {
        "root": "/data/wildlife_datasets/data/AnimalCLEF2026",
        "score": score_2026,
        "metric": "mean",
        "replace": {},
        "count_fn": overshoot_individuals_2026,
        "plot_fn": None,
        "plot_name": "AnimalCLEF2026: clustering",
        "score_col": "{}",
        "y_label": None,
    },
}

cols = ["LynxID2025", "SalamanderID2025", "SeaTurtleID2022"]
plot_names = {
    "LynxID2025": "lynxes",
    "SalamanderID2025": "salamanders",
    "SeaTurtleID2022": "sea turtles",
}

In [ ]:
solutions_all = {}
submissions_all = {}
results_all = {}
for name, edition in editions.items():
    root = edition["root"]

    solution = pd.read_csv(f"{root}/solution.csv")
    idx_public = solution["Usage"] == "Public"
    idx_private = ~idx_public

    root_submission = f"{root}/submissions"
    submissions = load_submissions(root_submission, participants_replace=edition["replace"])
    public, private = evaluate_submissions(submissions, solution, edition["score"], idx_public)
    results = select_best(submissions, public, private, metric=edition["metric"])

    solutions_all[name] = solution
    submissions_all[name] = submissions
    results_all[name] = results

In [ ]:
rows = []
for name, edition in editions.items():
    solution = solutions_all[name]
    submissions = submissions_all[name]
    results = results_all[name]
    estimator = edition["count_fn"]
    for i in range(len(results)):
        submission = get_submission(results, submissions, i)
        for dataset, overshoot in estimator(submission, solution).items():
            rows.append(
                {
                    "edition": name,
                    "team": results.index[i],
                    "rank": results["rank"].iloc[i],
                    "dataset": dataset,
                    "overshoot": overshoot,
                }
            )
estimates = pd.DataFrame(rows)
estimates = estimates[estimates["dataset"].isin(cols)]

In [ ]:
import seaborn as sns

df = estimates.dropna(subset=["overshoot"]).copy()
df["species"] = df["dataset"].map(plot_names)
df["framework"] = df["edition"].map({name: edition["plot_name"] for name, edition in editions.items()})
order = [plot_names[col] for col in cols]

plt.figure(figsize=(10, 3))
sns.violinplot(data=df, x="species", y="overshoot", hue="framework", order=order, cut=0, inner=None, log_scale=2, alpha=0.5)
sns.stripplot(data=df, x="species", y="overshoot", hue="framework", order=order, dodge=True, alpha=0.4, palette="dark:black", legend=False, log_scale=2)
plt.axhline(1, linestyle="--", color="red")
plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{y:g}"))
plt.xlabel("")
plt.ylabel("ratio of predicted/true individuals")
plt.legend(title=None)
plt.grid(alpha=0.4)
plt.savefig("population_editions.png", dpi=300, bbox_inches="tight")


In [ ]:
name = "AnimalCLEF2025"
results = results_all[name]
plot_fn = editions[name]["plot_fn"]
solution = solutions_all[name]
submissions = submissions_all[name]
score_col = editions[name]["score_col"]

overshoot = {}
for i in range(len(results)):
    submission = get_submission(results, submissions, i)
    overshoot[results.index[i]] = plot_fn(submission, solution)
overshoot = pd.DataFrame(overshoot).T

In [ ]:
fig, axes = plt.subplots(1, len(cols), figsize=(10, 2.5), sharey=True)
for ax, col in zip(axes, cols):
    ax.scatter(results[score_col.format(col)], overshoot[col], alpha=0.7)

    ax.axhline(1, linestyle="--", color="red")
    ax.set_yscale("log", base=2)
    ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{y:g}"))
    ax.set_title(plot_names[col])
    ax.set_xlabel("competition metric")
    ax.set_xlim([0.3, 0.8])
    ax.set_ylim([0.7, 10])
    ax.grid(alpha=0.4)
axes[0].set_ylabel(editions[name]["y_label"])
plt.savefig("metric_vs_overshoot.png", dpi=300, bbox_inches="tight")